# Phase 1 Fine-Tuning — Vehicle Re-ID (500-vehicle probe run)

Fine-tunes the Phase 0 vehicle Re-ID model to fix its two diagnosed failure modes (motion-blur fragility, camera/background context bias).

See `docs/VEHICLE_REID_FINDINGS_AND_DECISIONS.md` for the full rationale and `docs/superpowers/specs/2026-08-31-vehicle-reid-phase1-design.md` for the design spec.

**Before running:** Runtime -> Change runtime type -> select **T4 GPU**.

**Checkpoints now save to your Google Drive every 10 epochs** (not just at the end, and not to Colab's temporary disk) — added after a real run lost 55/60 Stage 2 epochs to a GPU-quota disconnect. If your session disconnects, use the **Resume** section near the bottom instead of starting over.

## 1. Mount Google Drive and upload the training bundle

**On your local machine first**, from `AI Registry/vehicle-reid/training/`:
```bash
zip -r CLIP-ReID-with-data.zip CLIP-ReID/ convert_to_onnx.py -x "CLIP-ReID/__pycache__/*"
```
**Note:** `convert_to_onnx.py` must be included at the zip's top level (not just inside `CLIP-ReID/`) — step 7 needs it directly on `/content`'s Python path. A real run hit `ModuleNotFoundError: No module named 'convert_to_onnx'` at the conversion step because an earlier zip build omitted it; the command above fixes that.

Upload `CLIP-ReID-with-data.zip` to the root of your Google Drive (drive.google.com -> New -> File upload). Also keep `blur_augmentation.py` and `../src/degrade.py` handy — they upload directly through Colab below, since they're tiny.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.chdir('/content')
print(os.listdir('/content/drive/MyDrive'))
# Confirm CLIP-ReID-with-data.zip appears in this listing before continuing.

In [ ]:
os.makedirs('/content/src', exist_ok=True)
from google.colab import files

print("Upload blur_augmentation.py now:")
uploaded = files.upload()
for name in uploaded:
    os.rename(name, f'/content/{name}')

print("Upload src/degrade.py now:")
uploaded = files.upload()
for name in uploaded:
    os.rename(name, f'/content/src/{name}')

open('/content/src/__init__.py', 'a').close()

## 2. Extract and verify the layout

In [ ]:
!rm -rf /content/CLIP-ReID
!unzip -q /content/drive/MyDrive/CLIP-ReID-with-data.zip -d /content/

In [ ]:
checks = [
    '/content/CLIP-ReID/datasets/veriwild.py',
    '/content/CLIP-ReID/configs/VehicleID/vit_clipreid_veriwild.yml',
    '/content/CLIP-ReID/data/train_list_start0.txt',
    '/content/convert_to_onnx.py',  # needed later at step 7 — checked now so
                                     # a missing zip entry is caught before
                                     # spending an hour training, not after.
]
for path in checks:
    status = 'OK' if os.path.exists(path) else 'MISSING'
    print(f'{status}: {path}')

num_images = len(os.listdir('/content/CLIP-ReID/data/images'))
print(f'Vehicle identity folders in data/images/: {num_images}')
assert num_images > 0, "No images found — check the zip uploaded and extracted correctly."
assert os.path.exists('/content/convert_to_onnx.py'), (
    "convert_to_onnx.py is missing — rebuild the zip locally with "
    "'zip -r CLIP-ReID-with-data.zip CLIP-ReID/ convert_to_onnx.py ...' "
    "(see step 1's note), re-upload to Drive, and re-run from step 2."
)

# Verify PIDs/camera IDs are the fixed, contiguous ranges (real bugs found
# and fixed on 2026-09-01 — see the config file's own header comment).
with open('/content/CLIP-ReID/data/train_list_start0.txt') as f:
    lines = [l for l in f.read().splitlines() if l]
pids = sorted(set(int(line.split(' ')[1]) for line in lines))
camids = sorted(set(int(line.split(' ')[2]) for line in lines))
print('PIDs contiguous 0..N-1:', pids == list(range(len(pids))))
print('Camera IDs contiguous 0..N-1:', camids == list(range(len(camids))))
assert pids == list(range(len(pids))) and camids == list(range(len(camids))), (
    "Data is not properly relabeled — do NOT proceed to training, this will crash."
)

## 3. Install dependencies

In [ ]:
!pip install -q torch torchvision yacs timm scikit-image tqdm ftfy regex opencv-python onnx onnxscript

## 4. Confirm GPU is actually available

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), "No GPU detected — set Runtime > Change runtime type > T4 GPU, then re-run this notebook from the top."

## 5a. Train — fresh start

Use this ONLY if you have never started this run before, or want to start over from scratch. If you're picking back up after a disconnect, skip to **5b. Resume** instead.

Checkpoints now save to `/content/drive/MyDrive/veriwild_finetune_output/` every 10 epochs (Stage 2) — they survive a disconnect.

In [ ]:
%cd /content/CLIP-ReID
import time
_start_time = time.time()
!python train_clipreid.py --config_file configs/VehicleID/vit_clipreid_veriwild.yml
_elapsed = time.time() - _start_time
print(f'\n\n=== Total training time: {_elapsed/60:.1f} minutes ===')

## 5b. Resume after a disconnect

Use this instead of 5a if a previous run got cut off (GPU quota, disconnect, etc.). This skips Stage 1 entirely and Stage 2 continues from the last saved checkpoint's epoch, with the learning-rate schedule fast-forwarded to match — not restarted.

**First, find your latest checkpoint:**

In [ ]:
!ls -la /content/drive/MyDrive/veriwild_finetune_output/

**Set `RESUME_CHECKPOINT` below to the highest-numbered `ViT-B-16_<epoch>.pth` file you see above**, then run:

In [ ]:
RESUME_CHECKPOINT = '/content/drive/MyDrive/veriwild_finetune_output/ViT-B-16_50.pth'  # EDIT THIS

assert os.path.exists(RESUME_CHECKPOINT), f"Not found: {RESUME_CHECKPOINT} — check the ls output above."

%cd /content/CLIP-ReID
import time
_start_time = time.time()
!python train_clipreid.py --config_file configs/VehicleID/vit_clipreid_veriwild.yml --resume_from "{RESUME_CHECKPOINT}"
_elapsed = time.time() - _start_time
print(f'\n\n=== Resumed training time this session: {_elapsed/60:.1f} minutes ===')

## 6. Locate the final checkpoint

In [ ]:
!ls -la /content/drive/MyDrive/veriwild_finetune_output/

## 7. Convert to ONNX

`NUM_CLASSES=500` and `CAMERA_NUM=114` are the real, already-verified values for this 500-vehicle probe run. If the dataset statistics table printed by training in step 5a/5b showed different numbers, **stop and use those instead**.

**Set `CHECKPOINT_FILENAME` below** to the highest-numbered file from the previous cell's output (should be `ViT-B-16_60.pth` if training completed all 60 epochs).

In [ ]:
import sys
sys.path.insert(0, '/content')
sys.path.insert(0, '/content/CLIP-ReID')

from convert_to_onnx import convert_checkpoint_to_onnx
from model.make_model_clipreid import make_model
from config import cfg

cfg.merge_from_file('/content/CLIP-ReID/configs/VehicleID/vit_clipreid_veriwild.yml')
cfg.freeze()

NUM_CLASSES = 500
CAMERA_NUM = 114
VIEW_NUM = 1  # VeriWild has no real viewpoint annotation — always 1

# EDIT THIS to the real filename printed by the previous cell's ls output:
CHECKPOINT_FILENAME = 'ViT-B-16_60.pth'
CHECKPOINT_PATH = f'/content/drive/MyDrive/veriwild_finetune_output/{CHECKPOINT_FILENAME}'

assert os.path.exists(CHECKPOINT_PATH), (
    f"Checkpoint not found at {CHECKPOINT_PATH} — check the real filename "
    f"from the ls output above and update CHECKPOINT_FILENAME."
)

INPUT_HEIGHT = 256
INPUT_WIDTH = 256

try:
    convert_checkpoint_to_onnx(
        checkpoint_path=CHECKPOINT_PATH,
        output_path='/content/vehicle_vit_clip_reid_finetuned.onnx',
        input_height=INPUT_HEIGHT,
        input_width=INPUT_WIDTH,
        model_factory=lambda: make_model(cfg, num_class=NUM_CLASSES, camera_num=CAMERA_NUM, view_num=VIEW_NUM),
    )
    print("Converted successfully.")
except RuntimeError as e:
    if "Missing key(s)" in str(e) or "Unexpected key(s)" in str(e):
        print("State dict key mismatch — likely a 'module.' prefix from DataParallel wrapping. Retrying with prefix stripped...")
        import torch as _torch
        state_dict = _torch.load(CHECKPOINT_PATH, map_location='cpu')
        stripped = {k.replace('module.', ''): v for k, v in state_dict.items()}
        stripped_path = '/content/checkpoint_stripped.pth'
        _torch.save(stripped, stripped_path)
        convert_checkpoint_to_onnx(
            checkpoint_path=stripped_path,
            output_path='/content/vehicle_vit_clip_reid_finetuned.onnx',
            input_height=INPUT_HEIGHT,
            input_width=INPUT_WIDTH,
            model_factory=lambda: make_model(cfg, num_class=NUM_CLASSES, camera_num=CAMERA_NUM, view_num=VIEW_NUM),
        )
        print("Converted successfully after stripping 'module.' prefix.")
    else:
        raise

## 8. Download the result

Save it locally into your project's `AI Registry/vehicle-reid/models/` folder as `vehicle_vit_clip_reid_finetuned.onnx`, then tell Claude — it will run `scripts/compare_before_after.py` for the real before/after comparison.

In [ ]:
from google.colab import files
files.download('/content/vehicle_vit_clip_reid_finetuned.onnx')